# Module 9 – DV01 Calculation

## Objective

This notebook demonstrates how to measure the sensitivity of an Interest Rate Swap to a 1 basis point parallel shift in the yield curve.

Pricing Workflow

Market Quotes
→ Bootstrapper
→ Yield Curve
→ Pricing Engine
→ Swap Valuation

Risk Workflow

Original Curve
        │
        ▼
   Price Swap
        │
        ▼
Parallel +1 bp Shift
        │
        ▼
 Reprice Swap
        │
        ▼
DV01 = PV(Shifted) − PV(Base)

For this stage, only the fixed leg is valued.

In [2]:
from datetime import date

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)
from src.curves.bootstrapper import Bootstrapper
from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection
from src.pricing.dv01_calculator import DV01Calculator
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap
from src.products.trade import Trade

## Step 1

Build the zero-coupon yield curve.

In [3]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.048))
quotes.add(MarketInstrument("Deposit", "3M", 0.050))
quotes.add(MarketInstrument("Deposit", "6M", 0.052))
quotes.add(MarketInstrument("Deposit", "1Y", 0.055))
quotes.add(MarketInstrument("Deposit", "2Y", 0.057))
quotes.add(MarketInstrument("Deposit", "3Y", 0.059))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

curve.summary()

Yield Curve
Valuation Date : 2026-01-01

Tenor        Years       Zero Rate       Discount Factor
------------------------------------------------------------------------
1M          0.0833        4.8000%              0.996008
3M          0.2500        5.0000%              0.987578
6M          0.5000        5.2000%              0.974335
1Y          1.0000        5.5000%              0.946485
2Y          2.0000        5.7000%              0.892258
3Y          3.0000        5.9000%              0.837780
------------------------------------------------------------------------
Interpolation : Linear
Curve Points  : 6


## Step 2

Create a vanilla fixed-for-floating interest rate swap.

In [4]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC Bank",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

## Step 3

Calculate the swap DV01 using a +1 basis point parallel shift in the yield curve.

In [5]:
result = DV01Calculator().calculate(
    swap,
    curve,
)

print(result)

DV01Result(trade_id='IRS001', currency='USD', present_value=133,940.91, dv01=-26.25)


## Step 4

Display the valuation and DV01.

In [6]:
print(f"Present Value : {result.present_value:,.2f} {result.currency}")
print(f"DV01          : {result.dv01:,.2f} {result.currency}")

Present Value : 133,940.91 USD
DV01          : -26.25 USD


## Summary

In this module, we introduced the first market risk measure.

Completed Workflow

Market Quotes
→ Yield Curve
→ Pricing Engine
→ Swap Valuation
→ DV01

DV01 measures the change in swap value resulting from a 1 basis point parallel shift in the yield curve.

The next module will extend this concept to **bucketed DV01 (key-rate risk)**, allowing us to identify which maturities contribute most to the swap's interest rate risk.